# Airline tweets: cleaning

**What this notebook does, in plain words**

You already have a file of airline tweets (`airline_tweets.csv`, made by the filter notebook). This notebook **cleans the tweets** (Part 2): it removes web links and signatures, hides personal details like phone numbers, marks tweets that are not useful, and links each customer complaint to the airline's reply.

It produces the cleaned files the next step (for example transformer labelling) reads: `tweets_clean.csv`, `pairs.csv`, `threads.csv` and `cleaning_report.json`.

You do **not** need to understand the code. You only need to run the cells in order.

### How to run it
- Click **Run All** at the top (Colab: *Runtime > Run all*; VS Code: *Run All*). Or click a cell and press **Shift + Enter**, cell by cell from top to bottom.
- A cell is finished when its output appears below it.

### Time needed
About a minute.

### Before you start: your checklist
- [ ] You have `airline_tweets.csv` (made earlier by the filter step, 370,863 rows).
- [ ] It is inside a folder called **`data`**.
- [ ] If you run on a laptop, you have `pandas`, `numpy` and `matplotlib` installed (`pip install pandas numpy matplotlib`). Google Colab already has them.

### Where do the files go?
```
airline_project/                    <- any folder you like; this notebook sits here too
    data/
        airline_tweets.csv          <- the filtered tweets
    airline_tweet_cleaning.ipynb    <- this notebook
```
- **On Google Colab:** make a folder called `airline_project` in your Google Drive (*My Drive*) and put the `data` folder inside it.
- **In VS Code or Jupyter:** put this notebook next to the `data` folder and open that folder.

The notebook creates the folder `outputs` by itself.

## Part 1: Settings and setup
**What happens:** the notebook works out where your files are.

**What you should see:** a few lines. `airline tweets` should say **found**.

In [ ]:
from pathlib import Path
import json, os

# ---- Settings (most people change nothing) -------------------------------------------
FORCE_RERUN = False   # Leave as False. True = redo the cleaning even if the finished files already exist.

# ---- Working out where your files are (no need to change this) -----------------------
try:
    from google.colab import drive          # this line only works on Google Colab
    drive.mount('/content/drive')           # Colab will ask you to allow access to your Drive
    BASE = Path('/content/drive/MyDrive/airline_project')
except ImportError:
    BASE = Path.cwd()                       # on a laptop: the folder this notebook is in

# If your files are somewhere else, remove the # at the start of the next line and put your folder path in it:
# BASE = Path(r"C:\Users\YourName\Desktop\airline_project")

INPUT = BASE / 'data' / 'airline_tweets.csv'     # the airline tweets (made by the filter notebook)
OUTDIR = BASE / 'outputs'                        # cleaned files go here
OUTPUT_FILES = ['tweets_clean.csv', 'pairs.csv', 'threads.csv', 'cleaning_report.json']

OUTDIR.mkdir(parents=True, exist_ok=True)
outputs_exist = all((OUTDIR / f).exists() for f in OUTPUT_FILES)

print('working folder  :', BASE.resolve())
print('airline tweets  :', 'found' if INPUT.exists() else 'NOT FOUND - see Troubleshooting at the bottom')
print('cleaned files   :', 'already exist (cleaning will be skipped)' if outputs_exist else 'not made yet')


## Part 2: Clean the tweets
**What happens:** every tweet is tidied:
- web links, `@mentions` and airline signatures (like `^Jane` or `*TJM`) are removed,
- **personal details are hidden** with tags like `<PHONE>`, `<EMAIL>`, `<BOOKING_REF>` and `<NAME>`,
- tweets that are not useful (one-word thank-yous, duplicates, non-English) are **marked**, not deleted,
- each complaint is **paired** with the airline's reply, and tweets are grouped into conversations.

**The first cell below only loads the cleaning rules and prints nothing. The second cell does the work.**

**What you should see:** a block of numbers (the report), then a list of saved files. If the files already exist, it says so and does nothing.

In [ ]:
#@title Cleaning rules (no need to read or change this; just run it)
"""Tweet cleaning for the airline NLP + RAG project.

Input : airline_tweets.csv (columns tweet_id, author_id, inbound, created_at, text,
        response_tweet_id, in_response_to_tweet_id)
Output: three DataFrames
        tweets : every tweet, with text_clean and flags
        pairs  : customer complaint -> first airline reply (the RAG knowledge base)
        threads: one row per conversation with a transcript (for summarisation)
        report : dict of counts to quote in the report

PII masking is heuristic. It removes the common patterns but cannot guarantee that no
personal detail survives, so say that in the report.
"""
import html
import re
import numpy as np
import pandas as pd

AIRLINE_HANDLES = ["AirAsiaSupport", "AlaskaAir", "AmericanAir", "British_Airways", "Delta",
                   "JetBlue", "SouthwestAir", "VirginAmerica", "VirginAtlantic"]
AIRLINE_LOWER = {h.lower() for h in AIRLINE_HANDLES}

# Words that follow "Hi" but are not first names (airline and brand words, plain greetings).
NOT_NAMES = {"there", "team", "all", "again", "guys", "everyone", "delta", "american", "british",
             "southwest", "jetblue", "alaska", "virgin", "airasia", "united", "air", "airways",
             "airlines", "support", "sir", "madam", "customer"}

# ---- regexes -------------------------------------------------------------------------
URL_RX = re.compile(r"https?://\S+|\bwww\.\S+", re.I)
EMAIL_RX = re.compile(r"__email__|\b[\w.+-]+@[\w-]+\.[\w.-]+\b", re.I)
LEAD_MENTIONS_RX = re.compile(r"^(?:\s*@\w+)+\s*")
MENTION_RX = re.compile(r"@(\w+)")
PHONE_RX = re.compile(r"\+?\d[\d\-\s().]{7,}\d")
LONG_NUM_RX = re.compile(r"\b[A-Za-z]{0,8}\d{7,}\b")   # 12345678, ATLWN01694804
CODE6_RX = re.compile(r"\b[A-Z0-9]{6}\b")
FLIGHT_LIKE_RX = re.compile(r"^[A-Z]{2}\d{3,4}$")
GREETING_RX = re.compile(r"\b((?i:hi|hello|hey|dear))\s+([A-Z][a-z]{2,})(?![A-Za-z0-9'])")   # name must be Capitalised
SIG_CARET_RX = re.compile(r"(?:\s*\d/\d)?\s*\^\s?[A-Za-z]{1,15}(?:\s*\d/\d)?\s*$")
SIG_STAR_RX = re.compile(r"(?:\s*\d/\d)?\s*\*\s?[A-Za-z]{2,5}(?:\s*\d/\d)?\s*$")   # Delta/American style: *TJM
SIG_DASH_RX = re.compile(r"\s*[-–—]\s?(?:[A-Z][a-z]{2,14}|[A-Z]{2,3})\s*$")
PART_MARK_RX = re.compile(r"(?<![\d/])\(?[1-4]/[2-4]\)?(?![\d/])")   # "1/2", "(2/2)" thread markers
WS_RX = re.compile(r"\s+")
ZERO_WIDTH_RX = re.compile(r"[​‌‍﻿]")

THANKS_RX = re.compile(r"\b(thank|thanks|thx|ty|cheers|appreciate)\b", re.I)
DM_RX = re.compile(r"\b(dm(?:'?ed)?|sent (?:you )?(?:a )?(?:dm|message|pm)|followed|following you|follow back|done)\b", re.I)

EN_STOP = set("""the a an and or but if so to of in on at for with from by is are was were be been am i you he she it we they
me my your our their this that these those not no yes do does did have has had will would can could should please
what when where why how who which just about as than then there here out up down over again more very too also""".split())


# ---- text cleaning -------------------------------------------------------------------
def _phone_sub(m):
    return "<PHONE>" if len(re.findall(r"\d", m.group(0))) >= 9 else m.group(0)


def _code_sub(m):
    s = m.group(0)
    if not (re.search(r"[A-Z]", s) and re.search(r"\d", s)):
        return s
    if FLIGHT_LIKE_RX.match(s):          # DL4047, BA0257 are flight numbers, keep them
        return s
    return "<BOOKING_REF>"


def _greeting_sub(m):
    first = m.group(2)
    if first.lower() in NOT_NAMES:
        return m.group(0)
    return f"{m.group(1)} <NAME>"


def _mention_sub(m):
    return "@airline" if m.group(1).lower() in AIRLINE_LOWER else "@user"


def clean_text(text, is_airline: bool = False) -> str:
    """Normalise one tweet and mask personal details."""
    if not isinstance(text, str):
        return ""
    s = html.unescape(text)
    s = ZERO_WIDTH_RX.sub("", s).replace("\xa0", " ")
    s = URL_RX.sub(" ", s)
    s = EMAIL_RX.sub(" <EMAIL> ", s)
    s = LEAD_MENTIONS_RX.sub("", s)
    s = MENTION_RX.sub(_mention_sub, s)
    if is_airline:
        for _ in range(2):                      # a tweet can carry both "2/2" and "^Name"
            s = SIG_CARET_RX.sub("", s)
            s = SIG_STAR_RX.sub("", s)
            s = SIG_DASH_RX.sub("", s)
        s = PART_MARK_RX.sub(" ", s)
    s = PHONE_RX.sub(_phone_sub, s)
    s = LONG_NUM_RX.sub("<NUMBER>", s)
    s = CODE6_RX.sub(_code_sub, s)
    s = GREETING_RX.sub(_greeting_sub, s)
    return WS_RX.sub(" ", s).strip()


def n_words(s: str) -> int:
    return len(re.findall(r"[A-Za-z']+", re.sub(r"<[A-Z_]+>", " ", s)))


def lang_guess(s: str) -> str:
    """Very rough: 'en', 'other' or 'short'. Good enough to flag Malay/Hindi/etc. tweets."""
    toks = re.findall(r"[a-z']+", re.sub(r"<[A-Z_]+>", " ", s.lower()))
    if len(toks) < 4:
        return "short"
    hits = sum(t in EN_STOP for t in toks) / len(toks)
    return "en" if hits >= 0.2 else "other"


def low_value_reason(s: str, words: int) -> str:
    """Why a CUSTOMER tweet is unlikely to be a useful complaint ('' = fine)."""
    if words < 4:
        return "too_short"
    if words <= 8 and THANKS_RX.search(s):
        return "thanks_only"
    if words <= 8 and DM_RX.search(s):
        return "dm_followup"
    return ""


# ---- main pipeline -------------------------------------------------------------------
def _norm_id(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x[:-2] if x.endswith(".0") else x


def run(path_or_df) -> dict:
    df = pd.read_csv(path_or_df, dtype=str) if not isinstance(path_or_df, pd.DataFrame) else path_or_df.astype(str)
    df = df.replace({"nan": np.nan})
    rep = {"rows_in": len(df)}

    df["tweet_id"] = df["tweet_id"].map(_norm_id)
    df["in_response_to_tweet_id"] = df["in_response_to_tweet_id"].map(_norm_id)
    df["inbound"] = df["inbound"].astype(str).str.lower().eq("true")
    before = len(df)
    df = df.drop_duplicates(subset="tweet_id")
    rep["dropped_duplicate_tweet_id"] = before - len(df)

    df["created_at"] = pd.to_datetime(df["created_at"], format="%a %b %d %H:%M:%S %z %Y", utc=True, errors="coerce")
    rep["unparsed_dates"] = int(df["created_at"].isna().sum())
    df["period"] = df["created_at"].dt.strftime("%Y-%m")
    df["is_core_window"] = (df["created_at"] >= "2017-10-01") & (df["created_at"] < "2018-01-01")

    df["text_raw"] = df["text"].fillna("")
    df["has_url"] = df["text_raw"].str.contains(URL_RX)
    is_air = ~df["inbound"]
    df["text_clean"] = [clean_text(t, a) for t, a in zip(df["text_raw"], is_air)]
    df["n_words"] = df["text_clean"].map(n_words)
    df["lang_guess"] = df["text_clean"].map(lang_guess)

    # masking counts for the report
    for tag in ["<EMAIL>", "<PHONE>", "<NUMBER>", "<BOOKING_REF>", "<NAME>"]:
        rep["masked_" + tag.strip("<>").lower()] = int(df["text_clean"].str.contains(re.escape(tag)).sum())

    # airline assignment
    df["author_lower"] = df["author_id"].str.lower()
    first_mention = df["text_raw"].str.extract(r"@(" + "|".join(AIRLINE_HANDLES) + r")\b", flags=re.I)[0]
    df["airline"] = np.where(df["author_lower"].isin(AIRLINE_LOWER), df["author_id"], first_mention)
    author_of = df.set_index("tweet_id")["author_id"]
    parent_author = df["in_response_to_tweet_id"].map(author_of)
    parent_air = parent_author.where(parent_author.str.lower().isin(AIRLINE_LOWER))
    df["airline"] = df["airline"].fillna(parent_air)
    df["airline"] = df["airline"].str.lower().map({h.lower(): h for h in AIRLINE_HANDLES})
    rep["customer_tweets_without_airline"] = int((df["inbound"] & df["airline"].isna()).sum())

    # customer-tweet quality flags
    cust = df["inbound"]
    df["low_value_reason"] = ""
    df.loc[cust, "low_value_reason"] = [low_value_reason(t, w) for t, w in
                                        zip(df.loc[cust, "text_clean"], df.loc[cust, "n_words"])]
    # "first copy" = earliest by time, so the result does not depend on the order of rows in the file
    key = df["text_clean"].str.lower()
    order = df[cust].assign(_id=pd.to_numeric(df.loc[cust, "tweet_id"], errors="coerce")) \
                    .sort_values(["created_at", "_id"]).index
    df["is_duplicate_text"] = False
    df.loc[order, "is_duplicate_text"] = key.loc[order].duplicated(keep="first").values
    df["keep_for_modelling"] = (cust & (df["low_value_reason"] == "") & ~df["is_duplicate_text"]
                                & (df["lang_guess"] == "en") & df["airline"].notna())

    # threads: follow in_response_to up to the root that is in the file
    parent = dict(zip(df["tweet_id"], df["in_response_to_tweet_id"]))
    ids = set(df["tweet_id"])
    root, depth = {}, {}

    def climb(t):
        path, seen = [], set()
        cur = t
        while cur not in root:
            if cur in seen:                       # cycle guard
                root[cur], depth[cur] = cur, 0
                break
            seen.add(cur)
            path.append(cur)
            p = parent.get(cur)
            if p is None or pd.isna(p) or p not in ids:
                root[cur], depth[cur] = cur, 0
                path.pop()
                break
            cur = p
        base_root, base_depth = root[cur], depth[cur]
        for i, node in enumerate(reversed(path)):
            root[node], depth[node] = base_root, base_depth + i + 1

    for t in df["tweet_id"]:
        climb(t)
    df["thread_id"] = df["tweet_id"].map(root)
    df["turn"] = df["tweet_id"].map(depth)
    df["thread_size"] = df.groupby("thread_id")["tweet_id"].transform("size")

    keep_cols = ["tweet_id", "author_id", "inbound", "created_at", "period", "is_core_window", "airline",
                 "thread_id", "turn", "thread_size", "text_raw", "text_clean", "n_words", "lang_guess",
                 "has_url", "low_value_reason", "is_duplicate_text", "keep_for_modelling",
                 "in_response_to_tweet_id", "response_tweet_id"]
    tweets = df[keep_cols].copy()

    # pairs: customer tweet -> earliest airline reply to it
    air_rows = df[~df["inbound"] & df["author_lower"].isin(AIRLINE_LOWER)]
    air_rows = air_rows.dropna(subset=["in_response_to_tweet_id"]).sort_values(["created_at", "tweet_id"])
    first_reply = air_rows.drop_duplicates(subset="in_response_to_tweet_id", keep="first")
    c = df[df["inbound"]]
    pairs = c.merge(first_reply, left_on="tweet_id", right_on="in_response_to_tweet_id", suffixes=("_c", "_r"))
    pairs = pd.DataFrame({
        "pair_id": np.arange(1, len(pairs) + 1),
        "airline": pairs["airline_r"],
        "complaint_id": pairs["tweet_id_c"], "reply_id": pairs["tweet_id_r"],
        "complaint": pairs["text_clean_c"], "reply": pairs["text_clean_r"],
        "complaint_time": pairs["created_at_c"], "reply_time": pairs["created_at_r"],
        "response_minutes": (pairs["created_at_r"] - pairs["created_at_c"]).dt.total_seconds() / 60,
        "complaint_words": pairs["n_words_c"], "reply_words": pairs["n_words_r"],
        "complaint_ok": pairs["keep_for_modelling_c"],
        "reply_lang": pairs["lang_guess_r"],
        "thread_id": pairs["thread_id_c"],
    })
    pairs["use_for_rag"] = pairs["complaint_ok"] & (pairs["reply_lang"] == "en") & (pairs["reply_words"] >= 5)
    rep["pairs_total"] = len(pairs)
    rep["pairs_use_for_rag"] = int(pairs["use_for_rag"].sum())

    # threads: transcript for summarisation
    t2 = df.sort_values(["thread_id", "created_at", "tweet_id"]).copy()
    t2["speaker"] = np.where(t2["inbound"], "Customer", "Airline")
    t2["line"] = t2["speaker"] + ": " + t2["text_clean"]
    g = t2.groupby("thread_id")
    threads = pd.DataFrame({
        "thread_id": g.size().index,
        "airline": g["airline"].agg(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan).values,
        "n_turns": g.size().values,
        "n_customer": g["inbound"].sum().values,
        "start": g["created_at"].min().values, "end": g["created_at"].max().values,
        "transcript": g["line"].agg("\n".join).values,
    })
    threads["duration_hours"] = (threads["end"] - threads["start"]).dt.total_seconds() / 3600
    threads["use_for_summary"] = threads["n_turns"].between(2, 20)   # very long threads are announcements, not support chats
    threads = threads[(threads["n_customer"] >= 1) & (threads["n_turns"] - threads["n_customer"] >= 1)]
    threads = threads.reset_index(drop=True)
    rep["threads_with_customer_and_airline"] = len(threads)
    rep["threads_use_for_summary"] = int(threads["use_for_summary"].sum())
    rep["empty_clean_text"] = int((df["text_clean"].str.len() == 0).sum())

    rep["rows_out"] = len(tweets)
    rep["customer_tweets"] = int(tweets["inbound"].sum())
    rep["airline_tweets"] = int((~tweets["inbound"]).sum())
    rep["low_value_customer"] = tweets.loc[tweets["inbound"], "low_value_reason"].replace("", np.nan).value_counts().to_dict()
    rep["duplicate_customer_text"] = int(tweets["is_duplicate_text"].sum())
    rep["customer_non_english_or_short"] = int((tweets["inbound"] & (tweets["lang_guess"] != "en")).sum())
    rep["keep_for_modelling"] = int(tweets["keep_for_modelling"].sum())
    rep["keep_in_core_window"] = int((tweets["keep_for_modelling"] & tweets["is_core_window"]).sum())
    return {"tweets": tweets, "pairs": pairs, "threads": threads, "report": rep}


In [ ]:
import pandas as pd
if outputs_exist and not FORCE_RERUN:
    print('The cleaned files are already in the outputs folder, so nothing was redone.')
    print('(To rebuild them, set FORCE_RERUN = True in Part 1 and run everything again.)\n')
    report = json.loads((OUTDIR / 'cleaning_report.json').read_text())
else:
    if not INPUT.exists():
        print('PROBLEM: I cannot find the airline tweets file at:')
        print('   ', INPUT.resolve())
        print('WHAT TO DO: put airline_tweets.csv in the data folder (see "Where do the files go?" at the top),')
        print('            or set BASE in Part 1 to the folder that holds it. Then run this cell again.')
        raise FileNotFoundError(str(INPUT))
    print('Cleaning the tweets. About one minute...')
    out = run(INPUT)
    tweets, pairs, threads, report = out['tweets'], out['pairs'], out['threads'], out['report']
    tweets.to_csv(OUTDIR / 'tweets_clean.csv', index=False)
    pairs.to_csv(OUTDIR / 'pairs.csv', index=False)
    threads.to_csv(OUTDIR / 'threads.csv', index=False)
    (OUTDIR / 'cleaning_report.json').write_text(json.dumps(report, indent=1, default=str))
    print('Saved these files in', OUTDIR.resolve())
    for n in OUTPUT_FILES:
        print(f"   {n:24s}{(OUTDIR / n).stat().st_size / 1e6:8.1f} MB")
print()
print(json.dumps(report, indent=1, default=str))


### The result in plain English

In [ ]:
r = report
print(f"The airline file has {r['rows_out']:,} tweets: {r['customer_tweets']:,} from customers and {r['airline_tweets']:,} from airlines.")
print(f"{r['keep_for_modelling']:,} customer tweets are good enough for the models (the column keep_for_modelling).")
print(f"{r['pairs_total']:,} complaint-and-reply pairs were built; {r['pairs_use_for_rag']:,} of them are good enough for the reply assistant.")
print(f"{r['threads_with_customer_and_airline']:,} conversations were rebuilt; {r['threads_use_for_summary']:,} are a good length for summaries.")
print(f"Personal details hidden: {r['masked_phone']:,} phone numbers, {r['masked_email']:,} emails, {r['masked_booking_ref']:,} booking codes, {r['masked_name']:,} first names.")


### Check that the cleaning worked
Each line counts problems left behind. **0 or a very small number (a few dozen) is fine.** If a line shows thousands, tell the project lead and send a screenshot.

Below you will see 8 examples: the original tweet, an arrow, then the cleaned version. Read a few. Do they look sensible?

*(This only runs when the data was just rebuilt.)*

In [ ]:
if 'tweets' in globals():
    c, air = tweets['text_clean'], ~tweets['inbound']
    checks = {
        'web links left': c.str.contains(r'https?://'),
        'unhidden @names left': c.str.contains(r'@(?!user\b|airline\b)\w+'),
        'long numbers left (could be phone/case numbers)': c.str.contains(r'\d{7,}'),
        'airline signatures like *TJM left': air & c.str.contains(r'\*\s?[A-Za-z]{2,5}\s*$'),
        'airline signatures like ^Jane left': air & c.str.contains(r'\^\s?\w+\s*$'),
    }
    for k, m in checks.items():
        print(f'{k:50s}{int(m.sum()):>8,}')
    print('\nEXAMPLES (original -> cleaned):\n')
    for _, r2 in tweets.sample(8, random_state=1).iterrows():
        print('CUSTOMER' if r2.inbound else 'AIRLINE ', '|', r2.text_raw[:140].replace('\n', ' '))
        print('        ->', r2.text_clean[:140])
        print()
else:
    print('Skipped: the files already existed, so nothing was rebuilt in this run.')


## Part 3: What the cleaned data looks like
**What happens:** five simple charts. Each title says what the chart shows. They read the files Part 2 saved, so you can run this part again at any time.

**Why look at them:** they show what the data can and cannot support (one quarter of data, uneven airlines, slow and fast responders), and they make a good starting point for the report.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

GREY, ACCENT, INK, SOFT = '#B9B7AB', '#2A78D4', '#222222', '#666666'
plt.rcParams.update({'font.size': 11, 'axes.edgecolor': '#999999', 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 13, 'axes.titleweight': 'normal',
                     'axes.titlelocation': 'left', 'figure.dpi': 110})

def finish(ax, name):
    """Save the chart as a picture and show it."""
    plt.tight_layout()
    plt.savefig(OUTDIR / name, dpi=150, bbox_inches='tight')
    plt.show()

tw = pd.read_csv(OUTDIR / 'tweets_clean.csv', dtype={'tweet_id': str},
                 usecols=['tweet_id', 'airline', 'inbound', 'period', 'low_value_reason',
                          'is_duplicate_text', 'lang_guess', 'keep_for_modelling'])
pr = pd.read_csv(OUTDIR / 'pairs.csv', usecols=['airline', 'response_minutes'])
th = pd.read_csv(OUTDIR / 'threads.csv', usecols=['n_turns'])
cust = tw[tw['inbound']]
print(f'{len(tw):,} tweets, {len(cust):,} of them from customers, {len(pr):,} complaint-and-reply pairs, {len(th):,} conversations')


In [ ]:
# Chart 1: tweets per month (the few tweets before September 2017 are grouped together)
label = tw['period'].where(tw['period'] >= '2017-09', 'Before Sep 2017')
m = tw.groupby(label).size()
order = ['Before Sep 2017'] + sorted(k for k in m.index if k != 'Before Sep 2017')
m = m.reindex(order)
in_core = [k in ('2017-10', '2017-11', '2017-12') for k in m.index]
share = m[in_core].sum() / m.sum()
fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.bar(m.index, m.values, color=[ACCENT if c else GREY for c in in_core], width=0.6)
for b, v in zip(bars, m.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v:,}', ha='center', va='bottom', color=INK, fontsize=10)
ax.set_ylim(0, m.max() * 1.12)
ax.set_title(f'{share:.1%} of the tweets were posted from October to December 2017')
ax.set_yticks([]); ax.spines['left'].set_visible(False)
finish(ax, 'chart_1_tweets_per_month.png')


In [ ]:
# Chart 2: customer tweets per airline, before and after cleaning
c = cust[cust['airline'].notna()]
raw = c.groupby('airline').size().sort_values()
kept = c[c['keep_for_modelling']].groupby('airline').size().reindex(raw.index).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(8, 4.6))
y = range(len(raw))
ax.barh([i + 0.2 for i in y], raw.values, height=0.38, color=GREY, label='All customer tweets')
ax.barh([i - 0.2 for i in y], kept.values, height=0.38, color=ACCENT, label='Kept for modelling')
for i, (r, k) in enumerate(zip(raw.values, kept.values)):
    ax.text(r, i + 0.2, f' {r:,}', va='center', fontsize=9, color=SOFT)
    ax.text(k, i - 0.2, f' {k:,}', va='center', fontsize=9, color=INK)
ax.set_yticks(list(y)); ax.set_yticklabels(raw.index); ax.set_xticks([]); ax.spines['bottom'].set_visible(False)
ax.set_title(f'{raw.index[-1]} has the most customer tweets and {raw.index[0]} the fewest')
ax.legend(frameon=False, loc='lower right')
finish(ax, 'chart_2_tweets_per_airline.png')


In [ ]:
# Chart 3: how fast each airline replies
med = pr.groupby('airline')['response_minutes'].median().sort_values()
overall = pr['response_minutes'].median()
fig, ax = plt.subplots(figsize=(8, 4.6))
slowest = sorted(med.index[-2:])
ax.barh(med.index, med.values, color=[ACCENT if a in slowest else GREY for a in med.index], height=0.6)
ax.axvline(overall, color=SOFT, linestyle='--', linewidth=1, zorder=0)
for i, v in enumerate(med.values):
    ax.text(v, i, f' {v:,.1f} min', va='center', fontsize=10, color=INK,
            bbox=dict(facecolor='white', edgecolor='none', pad=1))
ax.set_xticks([]); ax.spines['bottom'].set_visible(False)
ax.set_title(f'{slowest[0]} and {slowest[1]} take hours to reply;\nthe other airlines take minutes', pad=10)
ax.set_xlabel(f'Median minutes from the customer tweet to the first airline reply\n(dashed line = all airlines together, {overall:.1f} min)')
finish(ax, 'chart_3_reply_time.png')


In [ ]:
# Chart 4: how long conversations are
bins = pd.cut(th['n_turns'], [0, 2, 3, 4, 5, 10, 20, 10**6], labels=['2', '3', '4', '5', '6-10', '11-20', '21+'])
cnt = bins.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.bar(cnt.index.astype(str), cnt.values, color=[ACCENT if k == '2' else GREY for k in cnt.index], width=0.6)
for b, v in zip(bars, cnt.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v:,}', ha='center', va='bottom', fontsize=10, color=INK)
ax.set_title(f'The typical conversation is {int(th["n_turns"].median())} tweets long; the longest has {int(th["n_turns"].max()):,}')
ax.set_xlabel('Tweets in the conversation'); ax.set_yticks([]); ax.spines['left'].set_visible(False)
finish(ax, 'chart_4_conversation_length.png')


In [ ]:
# Chart 5: the cleaning funnel (customer tweets left after each filter)
s0 = cust
s1 = s0[s0['low_value_reason'].fillna('') == '']
s2 = s1[~s1['is_duplicate_text']]
s3 = s2[s2['lang_guess'] == 'en']
s4 = s3[s3['airline'].notna()]
steps = [('Customer tweets', len(s0)), ('Not too short, thanks-only or DM follow-up', len(s1)),
         ('Not a duplicate text', len(s2)), ('Judged English', len(s3)), ('Linked to an airline', len(s4))]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([n for n, _ in steps][::-1], [v for _, v in steps][::-1],
        color=[ACCENT] + [GREY] * 4, height=0.6)
for i, (n, v) in enumerate(steps[::-1]):
    ax.text(v, i, f' {v:,}', va='center', fontsize=10, color=INK)
ax.set_xticks([]); ax.spines['bottom'].set_visible(False)
ax.set_title(f'{len(s4):,} of {len(s0):,} customer tweets are left after cleaning')
finish(ax, 'chart_5_cleaning_funnel.png')
print('Matches keep_for_modelling:', len(s4) == int(cust['keep_for_modelling'].sum()))


### How to read them
- **Tweets per month:** the data is one quarter, so we cannot talk about seasons or longer trends.
- **Customer tweets per airline:** the airlines are very uneven, so a result for a small airline rests on few tweets.
- **Reply time:** a few airlines answer in minutes and others in hours. Any comparison of reply quality has to allow for this.
- **Conversation length:** most conversations are short, so thread summaries only apply to a minority.
- **Cleaning funnel:** how many customer tweets are left after each filter.

*Charts are saved as picture files in the `outputs` folder, so you can drop them straight into the report.*

## You are done when...
- Part 2 printed the report with no red errors and the four files are in `outputs`.
- Part 3 drew five charts and saved them as `chart_*.png` in `outputs`.

**What to send the group:** the files in `outputs`, and a screenshot of the plain-English result cell and the checks.

---

## What the new columns mean (for later)
| Column | Meaning |
| --- | --- |
| `text_clean` | The tweet after cleaning. **Use this, not `text_raw`.** |
| `keep_for_modelling` | `True` if a customer tweet is useful for models. **Filter on this when training or testing.** |
| `low_value_reason` | Why a customer tweet was marked not useful: `too_short`, `thanks_only` or `dm_followup`. |
| `is_duplicate_text` | The same customer text appeared earlier. |
| `lang_guess` | `en` (English), `other` or `short`. A rough guess, so spot-check it. |
| `thread_id`, `turn` | Which conversation a tweet belongs to, and its position in it. |
| `pairs.csv` | One row per complaint with the airline's first reply. `use_for_rag` marks the usable ones. |
| `threads.csv` | One row per conversation with a transcript. `use_for_summary` marks threads of 2 to 20 turns. |

## Words you may meet
- **Masking:** replacing a personal detail with a tag such as `<PHONE>`.
- **Flag:** a yes/no column that marks something, such as `keep_for_modelling`.
- **Thread / pair:** a conversation / a complaint plus the airline's first reply.

## Limits to mention in the report
- Hiding personal details is a best effort. It catches emails, phone numbers, booking codes, long case numbers and first names after "Hi/Hello/Dear", but it cannot promise nothing personal is left.
- Phone numbers are hidden everywhere, including airline support lines, so some replies say `<PHONE>`.
- The language check is a short word list, so some English tweets are marked not English and some other-language tweets may slip through.
- The tweets cover essentially October to December 2017 only.

---

## Troubleshooting
| What you see | What it means | What to do |
| --- | --- | --- |
| `airline tweets: NOT FOUND` | The notebook cannot find `airline_tweets.csv`. | Check it is called exactly `airline_tweets.csv` and sits in the `data` folder next to this notebook. Or set `BASE` in Part 1. |
| `MemoryError` or the notebook crashes | Your computer ran out of memory. | Close other programs and restart. On Colab use *Runtime > Restart session*. |
| Colab disconnected | Free sessions time out. | Reconnect and run again. Files saved in Drive are not lost, and finished steps are skipped. |
| Anything else | | Copy the red error text and send it to the project lead. |